# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [11]:
%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [12]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [31]:
import os
from glob import glob

PRICE_DATA = os.getenv("PRICE_DATA")
# os.path.join(price_dir, '*.parquet') #adding search for parquet file types to directory
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)



In [32]:
parquet_files

['../../05_src/data/prices\\A\\A_2000\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2001\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2002\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2003\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2004\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2005\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2006\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2007\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2008\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2009\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2010\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2011\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2012\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2013\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2014\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2015\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2016\\part.0.parquet',
 '../../05_src/data/prices\\A\\A_2017\\part.0.pa

In [33]:
dd_px = dd.read_parquet(parquet_files).set_index("Ticker")

c:\Users\long_\anaconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\core.py:5310: UserWarning: New index has same name as existing, this is a no-op.
  warnings.warn(


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [35]:
dd_shift = dd_px.groupby("Ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x["Close"].shift(1),
        Adj_Close_lag_2=x["Adj Close"].shift(1)))

dd_rets = dd_shift.assign(
    Returns = lambda x: x["Close"]/x["Close_lag_1"]-1)

dd_feat = dd_rets.assign(
    hi_lo_range = lambda x : x['High']- x['Low']
)

print(dd_feat)


<ipython-input-35-e303b2e718d4>:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby("Ticker", group_keys=False).apply(


Dask DataFrame Structure:
                                  Date Adj Close    Close     High      Low     Open   Volume   Year Close_lag_1 Adj_Close_lag_2  Returns hi_lo_range
npartitions=13078                                                                                                                                    
                   datetime64[ns, UTC]   float64  float64  float64  float64  float64  float64  int32     float64         float64  float64     float64
                                   ...       ...      ...      ...      ...      ...      ...    ...         ...             ...      ...         ...
...                                ...       ...      ...      ...      ...      ...      ...    ...         ...             ...      ...         ...
                                   ...       ...      ...      ...      ...      ...      ...    ...         ...             ...      ...         ...
                                   ...       ...      ...      ...      ..

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [41]:
# convert to pandas df
df_feat = dd_feat.compute()

#add moving average
df_feat['Returns'] = df_feat['Returns'].rolling(window=10).mean()

df_feat.head(n=20)

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_2,Returns,hi_lo_range
Ticker,,,,,,,,,,,,
DOV,2013-01-02 00:00:00+00:00,36.155922,45.073368,45.200733,44.765015,44.892380,2029431.0,2013,NaN,NaN,NaN,0.435719
DOV,2013-01-03 00:00:00+00:00,36.096767,44.999630,45.529198,44.858860,45.093479,1274735.0,2013,45.073368,36.155922,NaN,0.670338
DOV,2013-01-04 00:00:00+00:00,36.188198,45.113586,45.381721,44.966114,45.187325,1123169.0,2013,44.999630,36.096767,NaN,0.415607
DOV,2013-01-07 00:00:00+00:00,35.951588,44.818642,45.086777,44.697979,44.912487,943408.0,2013,45.113586,36.188198,NaN,0.388798
DOV,2013-01-08 00:00:00+00:00,35.887081,44.738201,45.428646,44.651058,45.240952,2040769.0,2013,44.818642,35.951588,NaN,0.777588
DOV,2013-01-09 00:00:00+00:00,36.150539,45.066666,45.153809,44.456657,44.825344,1621427.0,2013,44.738201,35.887081,NaN,0.697151
DOV,2013-01-10 00:00:00+00:00,36.247341,45.187325,45.428646,44.925896,45.361614,1571302.0,2013,45.066666,36.150539,NaN,0.502750
DOV,2013-01-11 00:00:00+00:00,36.064518,44.959412,45.173920,44.905785,45.093479,1024561.0,2013,45.187325,36.247341,NaN,0.268135
DOV,2013-01-14 00:00:00+00:00,35.811794,44.644352,44.992928,44.429848,44.838753,1793281.0,2013,44.959412,36.064518,NaN,0.563080


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?

No, moving average return can be caluclated with dask dataframe.

+ Would it have been better to do it in Dask? Why?

Yes, takes less time compared to pandas since it's more efficient with larger datasets.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.